[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C38_Frameworks_Accel_Course/02_torch_compile/02_torch_compile.ipynb)

# 02 · torch.compile 图捕获与编译（玩具编译器）

目标：从零写一个**玩具深度学习编译器**——用 tracing **捕获计算图**成 IR，再在图上做**算子融合**、**常量折叠**、**死代码消除**，每个 pass 之后都**对拍语义不变**。

路线：tracer（重载算子记录图）→ IR + `eval_graph` → 常量折叠 → 逐元素融合 → DCE → 串成编译流水 → ✏️ 练习（trace 记录 / 融合规则 / DCE）→ 📖 答案 → 🧪 真实数据胶囊（对照 torch.compile）。

> 心智模型：**编译 = 捕获成图 + 一串保语义的图变换 pass**。tracing 的本质是「用『记录』替换『执行』」。

## 1 · Tracer：用「记录」替换「执行」

tracing 的核心机制：让算子在被调用时**不直接算数值，而是把自己记录进一张图**。
我们用一个 `Tracer` 对象重载 `+ * relu` 等：每个运算返回一个新的 `Tracer`，同时往全局 `GRAPH` 追加一个节点。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# 一张图 = 节点列表；每个节点是一个 dict
class Graph:
    def __init__(self):
        self.nodes = []      # [{name, op, inputs:[names], const:val|None}]
        self._ctr = 0
    def fresh(self):
        n = f'%{self._ctr}'; self._ctr += 1; return n
    def add(self, op, inputs, const=None):
        name = self.fresh()
        self.nodes.append(dict(name=name, op=op, inputs=list(inputs), const=const))
        return name

class Tracer:
    '''一个被追踪的值：只携带它在图中的【名字】，不携带数值。'''
    def __init__(self, name, graph):
        self.name = name; self.graph = graph
    def _bin(self, other, op):
        if isinstance(other, Tracer):
            nm = self.graph.add(op, [self.name, other.name])
        else:                                  # 与常量运算
            cn = self.graph.add('const', [], const=float(other))
            nm = self.graph.add(op, [self.name, cn])
        return Tracer(nm, self.graph)
    def __add__(self, o): return self._bin(o, 'add')
    def __mul__(self, o): return self._bin(o, 'mul')
    def __radd__(self, o): return self._bin(o, 'add')
    def __rmul__(self, o): return self._bin(o, 'mul')
    def relu(self):
        return Tracer(self.graph.add('relu', [self.name]), self.graph)

def trace(fn, arg_names):
    '''跑一遍 fn（用 Tracer 占位输入），捕获出计算图。'''
    g = Graph()
    args = [Tracer(g.add('placeholder', [], const=nm), g) for nm in arg_names]
    # placeholder 的 const 字段借用来存参数名
    out = fn(*args)
    g.add('output', [out.name])
    return g

# 源程序: y = relu(a + b) * 2.0
def program(a, b):
    return (a + b).relu() * 2.0

g = trace(program, ['a', 'b'])
for nd in g.nodes:
    extra = f"  const={nd['const']}" if nd['const'] is not None else ''
    print(f"{nd['name']:>4} = {nd['op']:<11} {nd['inputs']}{extra}")
assert [nd['op'] for nd in g.nodes].count('add') == 1
assert any(nd['op'] == 'relu' for nd in g.nodes)
assert g.nodes[-1]['op'] == 'output'
print('\n✅ tracing 成功：算子没有直接算数值，而是被记录成了一张图(IR)')

## 2 · 解释执行图 `eval_graph`

有了 IR，写一个解释器按拓扑序求值（节点已按生成顺序排好，天然是拓扑序）。
这个 `eval_graph` 是后面所有 pass 的**裁判**：每次优化后，都用它对拍「优化前 == 优化后」。

In [ ]:
def eval_graph(g, feed):
    '''feed: {参数名: numpy 值}。按节点顺序求值，返回 output 的值。'''
    env = {}                                   # 节点名 -> 数值
    out_val = None
    for nd in g.nodes:
        op, nm, ins = nd['op'], nd['name'], nd['inputs']
        if op == 'placeholder':
            env[nm] = feed[nd['const']]        # const 存的是参数名
        elif op == 'const':
            env[nm] = nd['const']
        elif op == 'add':
            env[nm] = env[ins[0]] + env[ins[1]]
        elif op == 'mul':
            env[nm] = env[ins[0]] * env[ins[1]]
        elif op == 'relu':
            env[nm] = np.maximum(env[ins[0]], 0)
        elif op == 'fused_elementwise':
            env[nm] = _eval_fused(nd, env)      # 见融合 pass
        elif op == 'output':
            out_val = env[ins[0]]
        else:
            raise ValueError(f'未知算子 {op}')
    return out_val

a = rng.standard_normal((3, 4)); b = rng.standard_normal((3, 4))
got = eval_graph(g, {'a': a, 'b': b})
ref = np.maximum(a + b, 0) * 2.0               # 直接算
assert np.allclose(got, ref)
print('✅ eval_graph 对拍直接计算通过 —— IR 的语义与源程序一致')

## 3 · Pass 一：常量折叠

只依赖常量的子表达式，编译期就能算掉。我们识别「所有输入都是 `const` 的 `add`/`mul`/`relu`」节点，**编译期求值**、替换成一个 `const` 节点，并把后续对它的引用改向新常量。

> 小坑：`2.0*3.0` 这种**纯 Python 标量**会被 Python 当场算掉，trace 根本看不到。要让图里真的出现 `const*const`，下面**手工构造**一张图来演示折叠（真实编译器里，这类常量节点来自形状推导、`1/sqrt(d)` 等被 trace 进图的常量运算）。

In [ ]:
def constant_fold(g):
    new = Graph(); new._ctr = 0
    const_val = {}        # 旧节点名 -> 编译期已知的常量值（若可知）
    remap = {}            # 旧节点名 -> 新节点名
    for nd in g.nodes:
        op, nm, ins = nd['op'], nd['name'], nd['inputs']
        ins_new = [remap[i] for i in ins]
        if op == 'const':
            n2 = new.add('const', [], const=nd['const'])
            const_val[nm] = nd['const']; remap[nm] = n2
        elif op in ('add', 'mul', 'relu') and all(i in const_val for i in ins):
            # 所有输入都是编译期常量 -> 直接算
            vals = [const_val[i] for i in ins]
            v = (vals[0]+vals[1]) if op=='add' else (vals[0]*vals[1]) if op=='mul' else max(vals[0],0.0)
            n2 = new.add('const', [], const=float(v))
            const_val[nm] = float(v); remap[nm] = n2
        else:
            n2 = new.add(op, ins_new, const=nd['const'])
            remap[nm] = n2
    return new

# 注意：Python 会把 `2.0*3.0` 当场算成 6.0（两个都不是 Tracer），所以 trace 看不到这个 mul。
# 要让【图里】真的出现 const*const，我们手工构造图：y = a + mul(const(2.0), const(3.0))
def build_const_graph():
    g = Graph()
    a  = g.add('placeholder', [], const='a')
    c2 = g.add('const', [], const=2.0)
    c3 = g.add('const', [], const=3.0)
    m  = g.add('mul', [c2, c3])        # const * const -> 可折叠
    s  = g.add('add', [a, m])
    g.add('output', [s])
    return g
g2 = build_const_graph()
n_mul_before = sum(nd['op']=='mul' for nd in g2.nodes)
g2f = constant_fold(g2)
n_mul_after = sum(nd['op']=='mul' for nd in g2f.nodes)
a = rng.standard_normal((5,))
assert np.allclose(eval_graph(g2, {'a': a}), a + 6.0)    # 原图
assert np.allclose(eval_graph(g2f, {'a': a}), a + 6.0)   # 折叠后语义不变
assert n_mul_after < n_mul_before, '2.0*3.0 应被折叠掉'
print(f'mul 节点: 折叠前 {n_mul_before} -> 折叠后 {n_mul_after}')
print('✅ 常量折叠：mul(const 2.0, const 3.0) 在编译期算成 6.0，运行时不再重复算；结果不变')

## 4 · Pass 二：逐元素融合

把**连续的逐元素算子链**打包成一个 `fused_elementwise` 节点，中间结果不落主存（在真实编译器里就是生成一个内核）。
我们用一个简单规则：找到一段输入输出都是逐元素、且中间结果只被链内使用的 `add/mul/relu` 序列，融成一个节点（记录子图，求值时一次性算）。

In [ ]:
ELEMWISE = {'add', 'mul', 'relu'}

def _eval_fused(nd, env):
    '''融合节点：按它保存的子图(subnodes)依次算。子图里引用的【外部输入】用原名，
       而本节点 inputs 是它们在新图里的名字，二者按 sub['ext'] 顺序一一对应。'''
    sub = nd['const']
    # 原外部名 -> 当前值（外部输入的新名字 nd['inputs'] 已在 env 里）
    local = {orig: env[newnm] for orig, newnm in zip(sub['ext'], nd['inputs'])}
    for sn in sub['subnodes']:
        op, nm, ins = sn['op'], sn['name'], sn['inputs']
        if op == 'add':  local[nm] = local[ins[0]] + local[ins[1]]
        elif op == 'mul': local[nm] = local[ins[0]] * local[ins[1]]
        elif op == 'relu': local[nm] = np.maximum(local[ins[0]], 0)
    return local[sub['out']]

def fuse_elementwise(g):
    # 1) 统计每个节点被引用次数；by_name 便于查节点
    uses, by_name = {}, {nd['name']: nd for nd in g.nodes}
    for nd in g.nodes:
        for i in nd['inputs']:
            uses[i] = uses.get(i, 0) + 1
    # 2) 找【链尾】(融合根)：逐元素节点，且其输出不是『恰好被一个逐元素算子消费』。
    #    满足者就是某条逐元素链的末端（被 output、被非逐元素、或被多处使用）。
    consumers = {nd['name']: [] for nd in g.nodes}
    for nd in g.nodes:
        for i in nd['inputs']:
            consumers[i].append(nd['op'])
    def is_root(nd):
        if nd['op'] not in ELEMWISE: return False
        cs = consumers[nd['name']]
        return not (len(cs) == 1 and cs[0] in ELEMWISE)   # 唯一消费者也是逐元素 -> 还能往下并，非根
    # 3) 从每个根向上游贪心吸收【只被它用一次】的逐元素生产者，得到一条链
    consumed = set()
    chains = {}      # 根名 -> 链(按依赖序，根在最后)
    for nd in g.nodes:
        if not is_root(nd) or nd['name'] in consumed:
            continue
        chain = []
        def collect(name):
            node = by_name[name]
            for i in node['inputs']:
                pn = by_name.get(i)
                if pn and pn['op'] in ELEMWISE and uses.get(i, 0) == 1 and pn['name'] not in consumed:
                    collect(pn['name'])
            chain.append(node)
        collect(nd['name'])
        if len(chain) >= 2:
            for sn in chain: consumed.add(sn['name'])
            chains[nd['name']] = chain
    # 4) 重建图：被融合的链在其根处发射一个 fused_elementwise；其余节点原样搬
    new = Graph(); remap = {}
    for nd in g.nodes:
        if nd['name'] in consumed and nd['name'] not in chains:
            continue                                     # 链中间节点：并入根，跳过
        if nd['name'] in chains:
            chain = chains[nd['name']]
            names_in_chain = {sn['name'] for sn in chain}
            ext = []
            for sn in chain:
                for i in sn['inputs']:
                    if i not in names_in_chain and i not in ext:
                        ext.append(i)
            sub = dict(subnodes=chain, out=nd['name'], ext=ext)   # ext: 外部输入的【原名】，与 inputs 顺序一致
            remap[nd['name']] = new.add('fused_elementwise', [remap.get(i, i) for i in ext], const=sub)
        else:
            remap[nd['name']] = new.add(nd['op'], [remap.get(i, i) for i in nd['inputs']], const=nd['const'])
    return new

# 源程序: y = relu(a + b) * c  (三个逐元素算子)
def prog3(a, b, c):
    return (a + b).relu() * c
g3 = trace(prog3, ['a', 'b', 'c'])
n_before = len([nd for nd in g3.nodes if nd['op'] in ELEMWISE])
g3f = fuse_elementwise(g3)
n_fused = sum(nd['op']=='fused_elementwise' for nd in g3f.nodes)
n_after = len([nd for nd in g3f.nodes if nd['op'] in ELEMWISE])
a,b,c = (rng.standard_normal((3,4)) for _ in range(3))
feed = {'a':a,'b':b,'c':c}
assert np.allclose(eval_graph(g3f, feed), eval_graph(g3, feed))   # 语义不变！
print(f'逐元素节点: 融合前 {n_before} -> 融合后剩 {n_after} 个 + {n_fused} 个融合节点')
assert n_fused == 1 and n_after == 0
print('✅ 融合：3 个逐元素算子 -> 1 个 fused 节点，结果逐位不变(中间结果不落主存)')

## 5 · Pass 三：死代码消除 (DCE)

删掉**没有被 output 依赖**的节点。方法：从 output 反向标记所有可达节点，未标记的删。
DCE 常作为融合/折叠后的收尾（某些节点被绕过后成了孤儿）。

In [ ]:
def dead_code_elim(g):
    by_name = {nd['name']: nd for nd in g.nodes}
    live = set()
    # 从 output 反向标记可达
    def mark(name):
        if name in live or name not in by_name: return
        live.add(name)
        for i in by_name[name]['inputs']:
            mark(i)
    for nd in g.nodes:
        if nd['op'] == 'output':
            live.add(nd['name']); mark(nd['inputs'][0])
    new = Graph()
    new.nodes = [nd for nd in g.nodes if nd['name'] in live]
    return new

# 造一个含死代码的图：算了 dead = a*a 但 output 只用 a+b
def prog_dead(a, b):
    dead = a * a          # 这条没人用
    return a + b
g4 = trace(prog_dead, ['a', 'b'])
n_before = len(g4.nodes)
g4d = dead_code_elim(g4)
n_after = len(g4d.nodes)
a,b = rng.standard_normal((4,)), rng.standard_normal((4,))
assert np.allclose(eval_graph(g4d, {'a':a,'b':b}), a + b)   # 语义不变
assert n_after < n_before, '死代码 a*a 应被删'
assert not any(nd['op']=='mul' for nd in g4d.nodes), 'mul(死代码) 应消失'
print(f'节点数: DCE 前 {n_before} -> 后 {n_after}（删掉了没人用的 a*a）')
print('✅ DCE：删除不被 output 依赖的节点，结果不变')

## 6 · 串成编译流水线

把三个 pass 串起来 = 一个玩具 `compile`。对一个稍复杂的程序跑全流程，验证**编译前后逐位一致**、且节点数下降。

In [ ]:
def compile_graph(fn, arg_names, feed):
    g0 = trace(fn, arg_names)
    ref = eval_graph(g0, feed)
    g = constant_fold(g0)
    g = fuse_elementwise(g)
    g = dead_code_elim(g)
    got = eval_graph(g, feed)
    assert np.allclose(got, ref), '编译必须保语义！'
    return g, len(g0.nodes), len(g.nodes)

# 复杂程序：含常量子表达式、逐元素链、死代码
def big_prog(a, b):
    scale = 2.0 * 0.5            # 常量 -> 折叠成 1.0
    dead  = b * b               # 死代码
    h = (a + b).relu()          # 逐元素
    return (h * scale + a).relu()   # 逐元素链

a,b = rng.standard_normal((6,5)), rng.standard_normal((6,5))
g, n0, n1 = compile_graph(big_prog, ['a','b'], {'a':a,'b':b})
print(f'编译: {n0} 个节点 -> {n1} 个节点')
assert n1 < n0
print('✅ 全流水线：折叠+融合+DCE 后节点更少、结果逐位不变 —— 这就是 torch.compile 的骨架')

---
## ✏️ 练习 1：给 tracer 加 `square` 算子并捕获

给 `Tracer` 加一个方法 `square()`（前向 x*x），并给 `eval_graph` 加对应分支。

实现 `add_square_support()`：它给 `Tracer` 动态绑定 `square` 方法（记录一个 `square` 节点），并返回一个能 eval `square` 的 `eval_graph2`。然后 trace `lambda a: a.square() + 1.0` 并求值对拍。

In [ ]:
def add_square_support():
    # TODO: 1) 给 Tracer 加 square 方法：return Tracer(self.graph.add('square',[self.name]), self.graph)
    #       2) 定义 eval_graph2(g, feed)：在 eval_graph 基础上支持 op=='square' -> x*x
    #          (最简单：复制 eval_graph 的逻辑并加一个 elif)
    #       返回 eval_graph2
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
eval_graph2 = add_square_support()
gq = trace(lambda a: a.square() + 1.0, ['a'])
assert any(nd['op']=='square' for nd in gq.nodes), '应记录 square 节点'
a = rng.standard_normal((4,))
assert np.allclose(eval_graph2(gq, {'a':a}), a*a + 1.0)
print('✅ 练习 1 通过：tracer 与解释器都支持了新算子 square')

## ✏️ 练习 2：公共子表达式消除 (CSE)

同一个子表达式被算了多次时，只算一次、复用。给定一张图，实现 `cse(g)`：
若两个节点的 `(op, inputs, const)` 完全相同，把后者重定向到前者（去重）。

用 `a*b + a*b` 测试（`a*b` 出现两次，应被合并成一次）。

In [ ]:
def cse(g):
    # TODO: 遍历节点，用 (op, tuple(inputs_after_remap), const_key) 做 key 查重；
    #       已见过则 remap 到旧节点；否则保留。注意 const 可能是 dict(融合)，
    #       这里只需处理 add/mul/relu/const/placeholder，const_key 用 str(const)。
    #       返回去重后的新 Graph。
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
g_cse = trace(lambda a, b: (a*b) + (a*b), ['a','b'])
n_mul_before = sum(nd['op']=='mul' for nd in g_cse.nodes)
g_cse2 = cse(g_cse)
n_mul_after = sum(nd['op']=='mul' for nd in g_cse2.nodes)
a,b = rng.standard_normal((4,)), rng.standard_normal((4,))
assert np.allclose(eval_graph(g_cse2, {'a':a,'b':b}), a*b + a*b)
assert n_mul_after == 1 and n_mul_before == 2, f'a*b 应只算一次 (got {n_mul_after})'
print(f'mul 节点: CSE 前 {n_mul_before} -> 后 {n_mul_after}')
print('✅ 练习 2 通过：公共子表达式 a*b 只算一次')

## ✏️ 练习 3：统计编译收益（省了多少次主存往返）

融合的价值是减少中间结果的主存读写。实现 `mem_traffic(g)`：估算一张图的主存往返次数 = 
**每个非 fused 的逐元素/relu 算子算 1 次写 + 其输入里非常量的算 1 次读**，融合节点整体算「读 ext_inputs + 写 1 次」。

（简化模型：只数张量级读写次数。）对比融合前后的 traffic。

In [ ]:
def mem_traffic(g):
    # TODO: 遍历节点。对 add/mul/relu：traffic += (非const输入数) + 1(写出)
    #       对 fused_elementwise：traffic += (输入数) + 1
    #       const/placeholder/output 不计。返回总次数(int)。
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
def chain(a, b, c):
    return ((a + b).relu() * c).relu()    # 4 个逐元素算子链
gc = trace(chain, ['a','b','c'])
gcf = fuse_elementwise(gc)
t_before = mem_traffic(gc)
t_after  = mem_traffic(gcf)
print(f'主存往返(估计): 融合前 {t_before} -> 融合后 {t_after}')
assert t_after < t_before, '融合应减少主存往返'
print(f'✅ 练习 3 通过：融合把主存往返从 {t_before} 降到 {t_after}（中间结果留在片上）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def add_square_support():
    def square(self):
        return Tracer(self.graph.add('square', [self.name]), self.graph)
    Tracer.square = square
    def eval_graph2(g, feed):
        env = {}; out = None
        for nd in g.nodes:
            op, nm, ins = nd['op'], nd['name'], nd['inputs']
            if op == 'placeholder': env[nm] = feed[nd['const']]
            elif op == 'const': env[nm] = nd['const']
            elif op == 'add': env[nm] = env[ins[0]] + env[ins[1]]
            elif op == 'mul': env[nm] = env[ins[0]] * env[ins[1]]
            elif op == 'relu': env[nm] = np.maximum(env[ins[0]], 0)
            elif op == 'square': env[nm] = env[ins[0]] * env[ins[0]]
            elif op == 'output': out = env[ins[0]]
        return out
    return eval_graph2

In [ ]:
# 练习 2 参考答案
def cse(g):
    new = Graph(); remap = {}; seen = {}
    for nd in g.nodes:
        ins_new = [remap[i] for i in nd['inputs']]
        key = (nd['op'], tuple(ins_new), str(nd['const']))
        if nd['op'] in ('add','mul','relu','const') and key in seen:
            remap[nd['name']] = seen[key]            # 复用旧节点
        else:
            n2 = new.add(nd['op'], ins_new, const=nd['const'])
            remap[nd['name']] = n2
            seen[key] = n2
    return new

In [ ]:
# 练习 3 参考答案
def mem_traffic(g):
    by_name = {nd['name']: nd for nd in g.nodes}
    t = 0
    for nd in g.nodes:
        if nd['op'] in ('add','mul','relu'):
            non_const = sum(1 for i in nd['inputs'] if by_name[i]['op'] != 'const')
            t += non_const + 1
        elif nd['op'] == 'fused_elementwise':
            t += len(nd['inputs']) + 1
    return t

---
## 🧪 真实数据胶囊：对照 torch.compile

我们的玩具编译器和真实 `torch.compile` 做的是同一件事：捕获成图、优化、保语义。
下面在同一个函数上，对拍「eager 执行」与「编译后执行」结果一致（这正是真实编译器的回归测试）。

**装了 torch 才实跑；没装则用 numpy 模拟同一对拍（不阻断）。**

In [ ]:
# 我们的玩具编译器：对一个函数验证 eager == compiled
def demo(a, b):
    return ((a + b).relu() * 2.0 + a).relu()
a, b = rng.standard_normal((8, 8)), rng.standard_normal((8, 8))
g0 = trace(demo, ['a', 'b'])
g_compiled, n0, n1 = compile_graph(demo, ['a', 'b'], {'a': a, 'b': b})
print(f'我们的编译器: {n0} 节点 -> {n1} 节点, eager==compiled 已 assert 通过')

**🧪 胶囊练习**：实现 `torch_compile_matches_eager()`：
- 若有 torch：定义 `f(x)=torch.relu((x+1)*2)`，分别用 eager 与 `torch.compile(f)` 跑同一输入，返回两者是否 `allclose`；
- 若没 torch：用我们的玩具编译器做等价对拍（eager==compiled），返回 True。

无论哪条路，结论都应是 True：**编译保语义**。学生骨架（不计入自动验证）：

In [ ]:
def torch_compile_matches_eager():
    # TODO: 有 torch -> eager vs torch.compile 对拍；没 torch -> 玩具编译器对拍。返回 bool
    raise NotImplementedError

In [ ]:
# 自测（学生填好后运行）
assert torch_compile_matches_eager() == True, '编译必须与 eager 结果一致'
print('✅ 胶囊通过：torch.compile(或玩具编译器) 与 eager 结果一致 —— 编译保语义')

In [ ]:
# 📖 胶囊参考答案
def torch_compile_matches_eager():
    try:
        import torch
        def f(x): return torch.relu((x + 1) * 2)
        x = torch.randn(8, 8)
        fc = torch.compile(f)
        return bool(torch.allclose(f(x), fc(x), atol=1e-5))
    except Exception:
        # 回退：用玩具编译器做等价对拍
        def f(a): return ((a + 1.0) * 2.0).relu()
        x = rng.standard_normal((8, 8))
        _, _, _ = compile_graph(f, ['a'], {'a': x})   # 内部已 assert 保语义
        return True

---
## 🔧 旁注：真实 torch.compile 怎么用、怎么看图

我们手写的「trace → 优化 pass → eval」，在 PyTorch 里就是一行 `torch.compile`（对照，**不依赖即可读**）：

```python
import torch
@torch.compile                       # 就这一行：默认 eager、首调时捕获+编译
def model(x):
    return torch.relu((x + 1) * 2)    # 这串逐元素会被 Inductor 融成 1 个 Triton 内核

# 看捕获出的 FX 图（== 我们的 IR）：
from torch.fx import symbolic_trace
print(symbolic_trace(model).graph)   # placeholder / call_function / output 节点

# 看 graph break 在哪（== 我们的 trace 覆盖不到的地方）：
torch._dynamo.explain(model)(x)
```

对应关系：我们的 `trace` ↔ TorchDynamo（字节码层捕获）；我们的融合/折叠/DCE pass ↔ TorchInductor 的图优化；我们的 `fused_elementwise` ↔ Inductor 在 GPU 上生成的 Triton 融合内核。

### 小结
- eager 的天花板：一次只见一个算子，中间结果反复落主存（访存受限白白浪费）。
- 编译 = **捕获成图(IR) + 一串保语义的图变换 pass**。tracing 的本质是「用记录替换执行」。
- tracing 简单但**烤死控制流**；scripting 保控制流但只覆盖子集；torch.compile 用 tracing+graph break 折中。
- 三大 pass：**融合**(逐元素链合一、中间不落主存)、**常量折叠**(编译期算掉纯常量)、**DCE**(删没人用的节点)。
- 铁律：优化只改「怎么算」、不改「算出什么」——每个 pass 后都对拍语义不变。

下一站：**模块 03 · JAX/XLA 函数式与变换** —— 另一条路线：纯函数 + 可组合的 grad/vmap/jit。